In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D


In [ ]:
keep_cols = ['normalized_key', 'merged_mondo_label', 'merged_umls_label', 'preclinical_doc_ids', 'pub_year', 'max_phase', 'preclinical_ids_before_latest_trial', 'clinical_doc_ids', 'phase', 'trial_start_year', 'min_trial_start_year',
       'min_relevant_clinical_year', 'fda_earliest_year','translation_status', 'preclinical_count','preclinical_count_before_latest_trial','clinical_count']
file_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_for_timeline_NEURO.csv" # from Translation_04_Preclinical_Associations_Drug_Disease
preclin_dataset_to_clinical = pd.read_csv(file_path)[keep_cols]
preclin_dataset_to_clinical.columns

In [ ]:
preclin_dataset_to_clinical.head()

In [ ]:
# Drop rows missing any required field
preclin_dataset_to_clinical_clean = preclin_dataset_to_clinical.dropna(
    subset=[
        "min_relevant_clinical_year",
        "merged_umls_label",
        "merged_mondo_label",
        "min_trial_start_year"
    ]
)

print("Rows before:", len(preclin_dataset_to_clinical))
print("Rows after :", len(preclin_dataset_to_clinical_clean))
print("Dropped    :", len(preclin_dataset_to_clinical) - len(preclin_dataset_to_clinical_clean))

# Build dictionary

dict_clinical_relevant_years = {
    f"{row['merged_umls_label'].strip().lower()} | {row['merged_mondo_label'].strip().lower()}":
    int(row["min_relevant_clinical_year"])
    for _, row in preclin_dataset_to_clinical_clean.iterrows()
}

print("Dictionary size:", len(dict_clinical_relevant_years))

In [ ]:
dict_min_trials_years = {
    f"{row['merged_umls_label'].strip().lower()} | {row['merged_mondo_label'].strip().lower()}":
    int(row["min_trial_start_year"])
    for _, row in preclin_dataset_to_clinical_clean.iterrows()
}

In [ ]:
import pandas as pd
import numpy as np
import ast

def _as_list(x):
    """Safe conversion for list-like columns that might be strings."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "":
        return []
    try:
        return ast.literal_eval(s)  # handles "[...]" strings
    except Exception:
        # last resort: split on | (if you ever stored as "a|b|c")
        return [v.strip() for v in s.split("|") if v.strip()]

def build_drug_year_wide(
    df: pd.DataFrame,
    drug_col: str = "merged_umls_label",
    disease_col: str = "merged_mondo_label",
    doc_ids_col: str = "preclinical_doc_ids",
    years_col: str = "pub_year",
    status_col: str = "translation_status",
    index_name: str = "drug_disease",
    year_min: int | None = None,
    year_max: int | None = None,
    keep_cols: list[str] | None = None,   # e.g. ["translation_status", "min_relevant_clinical_year"]
) -> pd.DataFrame:

    work = df.copy()

    # build key like "drug | disease"
    work[index_name] = (
        work[drug_col].astype(str).str.strip()
        + " | " +
        work[disease_col].astype(str).str.strip()
    )

    # keep extra columns (one value per key)
    if keep_cols is None:
        keep_cols = [status_col]

    meta = (
        work[[index_name] + [c for c in keep_cols if c in work.columns]]
        .drop_duplicates(subset=[index_name])
        .set_index(index_name)
    )

    # ensure list columns are lists
    work[doc_ids_col] = work[doc_ids_col].apply(_as_list)
    work[years_col] = work[years_col].apply(_as_list)

    # explode paired lists row-by-row
    rows = []
    for key, doc_ids, years in zip(work[index_name], work[doc_ids_col], work[years_col]):
        n = min(len(doc_ids), len(years))
        if n == 0:
            continue

        for doc_id, y in zip(doc_ids[:n], years[:n]):
            if y is None or (isinstance(y, float) and np.isnan(y)):
                continue
            try:
                year_int = int(float(y))
            except Exception:
                continue

            if year_min is not None and year_int < year_min:
                continue
            if year_max is not None and year_int > year_max:
                continue

            rows.append((key, year_int, doc_id))

    long_df = pd.DataFrame(rows, columns=[index_name, "year", "preclinical_doc_id"])

    counts = (
        long_df.drop_duplicates([index_name, "year", "preclinical_doc_id"])
              .groupby([index_name, "year"])
              .size()
              .reset_index(name="n")
    )

    wide = (
        counts.pivot(index=index_name, columns="year", values="n")
              .fillna(0)
              .astype(int)
              .sort_index(axis=1)
    )

    # attach translation_status (and any other keep_cols)
    wide = meta.join(wide, how="right")  # keeps all keys that appear in counts

    return wide

In [ ]:
wide_df = build_drug_year_wide(
    df=preclin_dataset_to_clinical_clean,
    drug_col="merged_umls_label",
    disease_col="merged_mondo_label",
    doc_ids_col="preclinical_doc_ids",
    years_col="pub_year",
    year_min=1977,
    year_max=2025,
)

wide_df.head()

## VIZ

In [ ]:
def _size_transform(vals, mode="absolute", scale=40, eps=1e-9):
    v = np.asarray(vals, dtype=float)
    if mode == "absolute":              # current behavior
        sz = v * scale
    elif mode == "sqrt":                 # good for wide ranges
        sz = np.sqrt(v) * scale
    elif mode == "log":                  # very aggressive compression
        sz = np.log1p(v) * scale
    elif mode == "row_percent":          # relative to per-drug max
        denom = v.max() if np.isfinite(v.max()) and v.max() > 0 else 1.0
        sz = (v / (denom + eps)) * scale
    else:
        raise ValueError("size_mode must be one of: absolute|sqrt|log|row_percent")
    return sz


In [ ]:
def wrap_drug_label(name: str, max_len: int = 25) -> str:
    """
    Wrap long drug labels onto two lines for plotting.
    - If there's a '|' (e.g. 'Drug | Disease'), split there.
    - Otherwise, break at the nearest space around the middle.
    """
    if not isinstance(name, str):
        return name

    name = name.strip()

    # Prefer splitting at '|'
    if "|" in name:
        left, right = name.split("|", 1)
        return left.strip() + "\n" + right.strip()

    # If it's short enough, keep as is
    if len(name) <= max_len:
        return name

    # Find a space near the middle to break on
    mid = len(name) // 2
    # Look left then right for a space
    break_pos = None
    for offset in range(0, mid):
        left = mid - offset
        right = mid + offset
        if left > 0 and name[left] == " ":
            break_pos = left
            break
        if right < len(name) and name[right] == " ":
            break_pos = right
            break

    if break_pos is None:
        # No space found, hard break
        break_pos = mid

    return name[:break_pos].rstrip() + "\n" + name[break_pos:].lstrip()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

def plot_drug_year_bubbles_from_wide_with_extra_studies(
    drug_years: dict,
    earliest_phase_years: dict,
    wide_df: pd.DataFrame,
    title: str = "Timeline of preclinical studies and clinical milestones",
    extra_studies_df: pd.DataFrame | None = None,
    output_file: str | None = None,
    drugs_custom_order: list | None = None,
    year_range: tuple[int, int] | None = None,
    scale: int | float = 40,
    extra_scale: int | float | None = None,
    normalize_extra_studies: bool = False,
    status_col: str = "translation_status",
    approved_value: str = "approved",
    star: str = "★",
    wrap_func=None,
    fda_nme_pairs: list[str] | None = None,
    fda_nme_star: str = "✦",
):
    """
    Bubble timeline from a wide table (counts per drug_disease-year).

    Parameters
    ----------
    drug_years
        Dict mapping "drug | disease" to a clinical relevant year.
    earliest_phase_years
        Dict mapping "drug | disease" to earliest clinical trial year.
    wide_df
        Wide table indexed by "drug | disease", with year columns and optionally `status_col`.
    extra_studies_df
        Optional wide table indexed by drug (lower/upper case allowed), with year columns.
        Used to show "other disease" background bubbles.
    scale
        Bubble size multiplier for main bubbles (raw study counts).
    extra_scale
        Bubble size multiplier for background bubbles. If None, defaults to:
        - `scale` when `normalize_extra_studies=False`
        - `scale * 1000` when `normalize_extra_studies=True`
    normalize_extra_studies
        If True, normalize both `extra_row` and `disease_row` by the total number of
        extra studies in each year before subtraction. This makes background bubbles
        represent relative yearly activity instead of raw counts.
    """

    # ---- Font sizes (+2 applied globally) ----
    FS_PANEL = 24
    FS_BUBBLE = 12
    FS_YTICKS = 20
    FS_XLABEL = 18
    FS_TITLE = 20
    FS_LEGEND = 16

    # ---- 0) Capture status for labels, then drop status column ----
    status_map = {}
    wide_df = wide_df.copy()
    if status_col in wide_df.columns:
        status_map = (
            wide_df[status_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .to_dict()
        )
        wide_df = wide_df.drop(columns=[status_col])

    fda_nme_pairs = set(fda_nme_pairs or [])

    # ---- 1) Detect year columns ----
    year_cols = []
    for c in wide_df.columns:
        try:
            year_cols.append(int(c))
        except (TypeError, ValueError):
            pass
    year_cols = sorted(set(year_cols))

    if not year_cols:
        raise ValueError("No year columns detected in `wide_df`.")

    if year_range:
        y0, y1 = year_range
        year_cols = [y for y in year_cols if y0 <= y <= y1]
        if not year_cols:
            raise ValueError("After applying year_range, no year columns remain.")

    # ---- 2) Ensure numeric counts in wide_df ----
    wide_df[year_cols] = (
        wide_df[year_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(int)
    )

    # ---- 3) Prepare extra_studies_df if provided ----
    year_totals = None

    if extra_studies_df is not None:
        extra_studies_df = extra_studies_df.copy()
        extra_studies_df.index = extra_studies_df.index.astype(str).str.lower()

        for y in year_cols:
            if y not in extra_studies_df.columns:
                extra_studies_df.loc[:, y] = 0

        extra_studies_df[year_cols] = (
            extra_studies_df[year_cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
            .astype(float)
        )

        if normalize_extra_studies:
            year_totals = extra_studies_df[year_cols].sum(axis=0).replace(0, np.nan)

    # Default extra scale
    if extra_scale is None:
        extra_scale = scale * 1000 if normalize_extra_studies else scale

    # ---- 4) Choose drugs to plot ----
    if drugs_custom_order:
        drugs_in_order = [d for d in drugs_custom_order if d in wide_df.index]
    else:
        drugs_in_order = list(wide_df.index)

    n_drugs = len(drugs_in_order)
    print(f"Plotting {n_drugs} drugs")
    if n_drugs == 0:
        raise ValueError("No drugs to plot.")

    # ---- 5) Figure and axes ----
    fig_height = max(5, 0.95 * n_drugs)
    plt.figure(figsize=(19, fig_height))
    ax = plt.gca()

    ax.text(
        -0.08, 1.02, "e",
        transform=ax.transAxes,
        fontsize=FS_PANEL,
        fontweight="bold",
        va="bottom",
        ha="left"
    )

    COLOR_MAIN = "#56B4E9"
    COLOR_BG = "grey"
    COLOR_TRIAL = "palegreen"   # purple → earliest clinical trial
    COLOR_CLIN  = "#E69F00"     # orange → clinically relevant year

    # ---- 6) Plot per drug ----
    for i, drug in enumerate(drugs_in_order):
        y_pos = n_drugs - 1 - i

        row = wide_df.loc[drug, year_cols]
        nonzero = row[row > 0]
        if nonzero.empty:
            continue

        # Main bubbles
        ax.scatter(
            nonzero.index.astype(int),
            np.full(len(nonzero), y_pos),
            s=(nonzero.values * scale),
            color=COLOR_MAIN,
            alpha=0.7,
            zorder=2,
        )

        # Background bubbles: other disease studies
        if extra_studies_df is not None:
            drug_lookup = drug.split("|")[0].strip() if "|" in drug else str(drug).strip()
            alias = {"Fampridine": "Dalfampridine"}
            drug_extra = alias.get(drug_lookup, drug_lookup)

            disease_row = (
                wide_df.loc[drug, year_cols].astype(float)
                if drug in wide_df.index
                else pd.Series(0.0, index=year_cols)
            )

            drug_key = str(drug_extra).strip().lower()

            extra_row = (
                extra_studies_df.loc[drug_key, year_cols].astype(float)
                if drug_key in extra_studies_df.index
                else pd.Series(0.0, index=year_cols)
            )

            if normalize_extra_studies:
                disease_row = disease_row.div(year_totals).fillna(0)
                extra_row = extra_row.div(year_totals).fillna(0)

            resid = (extra_row - disease_row).clip(lower=0)
            nonzero_resid = resid[resid > 0]

            if not nonzero_resid.empty:
                ax.scatter(
                    nonzero_resid.index.astype(int),
                    np.full(len(nonzero_resid), y_pos),
                    s=(nonzero_resid.values * extra_scale),
                    alpha=0.1,
                    color=COLOR_BG,
                    edgecolor="black",
                    linewidth=0.4,
                    zorder=1,
                )

        # Numbers on main bubbles
        if False:
            for yr, n in nonzero.items():
                ax.text(
                    int(yr), y_pos, int(n),
                    ha="center", va="center",
                    fontsize=FS_BUBBLE,
                    zorder=3
                )

        # Earliest clinical trial marker
        earliest_phase_year = earliest_phase_years.get(drug, None)
        if earliest_phase_year:
            try:
                ay = int(earliest_phase_year)
                ax.plot(
                    [ay, ay],
                    [y_pos - 0.4, y_pos + 0.4],
                    linestyle="--",
                    color="grey",
                    linewidth=2.0,
                    alpha=0.8,
                    zorder=3,
                )
                ax.scatter(
                    [ay], [y_pos],
                    marker="D",
                    s=80,
                    color=COLOR_TRIAL,
                    edgecolor="black",
                    linewidth=0.5,
                    zorder=5,
                )
            except (TypeError, ValueError):
                pass

        # Clinical relevant year marker
        appr_year = drug_years.get(drug, None)
        if appr_year:
            try:
                ay = int(appr_year)
                ax.plot(
                    [ay, ay],
                    [y_pos - 0.4, y_pos + 0.4],
                    linestyle="--",
                    color="grey",
                    linewidth=2.0,
                    alpha=0.8,
                    zorder=3,
                )
                ax.scatter(
                    [ay], [y_pos],
                    marker="s",
                    s=80,
                    color=COLOR_CLIN,
                    edgecolor="black",
                    linewidth=0.5,
                    zorder=5,
                )
            except (TypeError, ValueError):
                pass

    # ---- 7) Axis styling ----
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    def _wrap(x: str) -> str:
        return wrap_func(x) if callable(wrap_func) else x

    labels = []
    for d in reversed(drugs_in_order):
        base = _wrap(d)
        status = status_map.get(d, "").strip().lower()

        if d in fda_nme_pairs:
            base = f"{fda_nme_star} {base}"
        elif status == approved_value.lower():
            base = f"{star} {base}"

        labels.append(base)

    ax.set_yticks(range(n_drugs))
    ax.set_yticklabels(labels, fontsize=FS_YTICKS)

    ax.set_ylim(-1, n_drugs)
    ax.set_xlim(min(year_cols) - 0.5, max(year_cols) + 0.5)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)

    legend_elements = [
        Line2D([0], [0], marker="o", linestyle="None",
               markerfacecolor=COLOR_MAIN, markeredgecolor="black",
               markersize=10, label="Animal studies"),
        Line2D([0], [0], marker="D", linestyle="None",
               markerfacecolor=COLOR_TRIAL, markeredgecolor="black",
               markersize=8, label="Earliest clinical trial"),
        Line2D([0], [0], marker="s", linestyle="None",
               markerfacecolor=COLOR_CLIN, markeredgecolor="black",
               markersize=8, label="Clinical relevant year"),
    ]

    if extra_studies_df is not None:
        extra_label = (
            "Animal studies (other disease, normalized)"
            if normalize_extra_studies
            else "Animal studies (other disease)"
        )
        legend_elements.append(
            Line2D([0], [0], marker="o", linestyle="None",
                   markerfacecolor="grey", markeredgecolor="black",
                   alpha=0.1, markersize=10, label=extra_label)
        )

    if status_map:
        legend_elements.append(
            Line2D(
                [0], [0],
                linestyle="None",
                marker=r"$★$",
                color="black",
                markersize=12,
                label="translated"
            )
        )

    if fda_nme_pairs:
        legend_elements.append(
            Line2D(
                [0], [0],
                linestyle="None",
                marker=r"$✦$",
                color="black",
                markersize=12,
                label="FDA new molecular entity"
            )
        )

    ax.legend(
        handles=legend_elements,
        fontsize=FS_LEGEND,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0,
        frameon=True
    )

    plt.xlabel("Publication year", fontsize=FS_XLABEL)
    plt.title(title, fontsize=FS_TITLE)
    plt.xticks(fontsize=FS_YTICKS)
    plt.yticks(fontsize=FS_YTICKS)
    plt.tight_layout()

    if output_file:
        plt.savefig(output_file, dpi=300, bbox_inches="tight")
        print(f"Saved plot to: {output_file}")

    plt.show()

In [ ]:
# generated in Translation_02
all_drugs_timeline = pd.read_csv("./out/all_drugs_articles_timeline.csv")

def format_wide_to_indexed(
    df: pd.DataFrame,
    name_col: str = "merged_umls_label",
    index_name: str = "unique_drug_target"
) -> pd.DataFrame:
    out = df.copy()
  
    # Convert year-like columns to ints (keep 'Total' as-is)
    year_like = [c for c in out.columns if c not in [name_col, "Total"]]
    col_map = {}
    for c in year_like:
        try:
            col_map[c] = int(c)
        except (ValueError, TypeError):
            pass  # ignore non-year extras
    out = out.rename(columns=col_map)

    # Set index to drug name
    out = out.set_index(name_col)
    out.index.name = index_name

    # Determine ordered year columns
    year_cols = sorted([c for c in out.columns if isinstance(c, int)])

    # Ensure numeric and integer dtype
    out[year_cols] = out[year_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # Add/position Total at the end
    if "Total" not in out.columns:
        out["Total"] = out[year_cols].sum(axis=1)
    else:
        out["Total"] = pd.to_numeric(out["Total"], errors="coerce").fillna(0).astype(int)

    out = out[year_cols + ["Total"]]

    # Name the columns axis as 'year'
    out.columns.name = "year"
    return out
    
all_drugs_timeline = format_wide_to_indexed(all_drugs_timeline)


In [ ]:
all_drugs_timeline.head()

In [ ]:
wide_df[wide_df.index.str.contains("aducanumab", case=False, na=False)]

In [ ]:
# Detect year columns
year_cols = [c for c in wide_df.columns if str(c).isdigit()]

# Filter rows with >=15 total studies
wide_df_filtered = wide_df[wide_df[year_cols].sum(axis=1) >= 15].copy()
print("Rows with >=15 studies:", len(wide_df_filtered))

# Add name length
wide_df_filtered["name_len"] = wide_df_filtered.index.str.len()

# Keep only shorter names
wide_df_filtered = wide_df_filtered[wide_df_filtered["name_len"] <= 40]
print("Rows with short names:", len(wide_df_filtered))

fda_nme_pairs = [
    "siponimod | multiple sclerosis", # 2019
    #"aducanumab | alzheimer disease", # 2021
    "xanomeline | schizophrenia" #2024
]

# Sample rows + add required ones
sample_wide_df = pd.concat([
    wide_df_filtered[wide_df_filtered["translation_status"] == "approved"].sample(2, random_state=42),
    wide_df_filtered[wide_df_filtered["translation_status"] == "failed"].sample(3, random_state=42),
    wide_df_filtered.loc[wide_df_filtered.index.intersection(fda_nme_pairs)]
])

# Remove duplicates if a fda_nme_pairs was already sampled
sample_wide_df = sample_wide_df[~sample_wide_df.index.duplicated(keep="first")]

# Drop helper column
sample_wide_df = sample_wide_df.drop(columns=["name_len"])

sample_wide_df

In [ ]:
# keep only those that exist
fda_present = [d for d in fda_nme_pairs if d in sample_wide_df.index]

# remaining pairs
remaining = [d for d in sample_wide_df.index if d not in fda_present]

drugs_custom_order = fda_present + remaining

In [ ]:
top_n = 10
if top_n > len(sample_wide_df):
    top_n = len(sample_wide_df)
plot_drug_year_bubbles_from_wide_with_extra_studies(
    drug_years=dict_clinical_relevant_years,
    earliest_phase_years=dict_min_trials_years,
    wide_df=sample_wide_df.head(top_n),
    title="Timeline of Preclinical Studies and Clinical Milestones",
    extra_studies_df=all_drugs_timeline,
    output_file=f"viz/top_{top_n}_selected_drug_timeline_preclinical_clinical.pdf",
    drugs_custom_order=drugs_custom_order,  # optional
    year_range=(1979, 2024),
    scale=20,
    fda_nme_pairs=fda_nme_pairs,
    #normalize_extra_studies=True,
    #extra_scale=40000,
)

In [ ]:
"Preclinical study timelines for selected drug–disease pairs".title()

In [ ]:
dict_min_trials_years['ketamine | hyperalgesia']

In [ ]:
dict_clinical_relevant_years['ketamine | hyperalgesia']

In [ ]:
preclin_dataset_to_clinical_clean[preclin_dataset_to_clinical_clean['normalized_key']=="hyperalgesia <> ketamine"]

In [ ]:
preclin_dataset_to_clinical_clean[preclin_dataset_to_clinical_clean['merged_umls_label']=="phenytoin"].head()